# Processed 250Hz EDA (PySpark + MinIO)

This notebook fetches parquet keys under `processed/250hz/` from MinIO, then reads each dataset directly with Spark (`s3a://`) into its own DataFrame without downloading to local cache.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('boto3') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'boto3'])


In [ ]:
import os
from collections import defaultdict

import boto3
from botocore.config import Config

S3_ENDPOINT_URL = os.getenv('S3_ENDPOINT_URL', 'http://minio:9000')
S3_ACCESS_KEY = os.getenv('S3_ACCESS_KEY', 'minioadmin')
S3_SECRET_KEY = os.getenv('S3_SECRET_KEY', 'minioadmin')
S3_BUCKET = os.getenv('S3_BUCKET', 'ecg-pipeline')
PREFIX = 'processed/250hz/'

s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT_URL,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name='us-east-1',
    config=Config(s3={'addressing_style': 'path'})
)

parquet_keys = []
annotation_keys = []
keys_by_dataset = defaultdict(list)
annotation_keys_by_dataset = defaultdict(list)
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=PREFIX):
    for obj in page.get('Contents', []):
        key = obj['Key']
        relative_key = key[len(PREFIX):] if key.startswith(PREFIX) else key
        dataset_name = relative_key.split('/', 1)[0]
        if key.endswith('.parquet'):
            parquet_keys.append(key)
            keys_by_dataset[dataset_name].append(key)
        elif key.endswith('.annotations.json'):
            annotation_keys.append(key)
            annotation_keys_by_dataset[dataset_name].append(key)

print(f'Found {len(parquet_keys)} parquet files in s3://{S3_BUCKET}/{PREFIX}')
print(f'Found {len(annotation_keys)} annotation JSON files in s3://{S3_BUCKET}/{PREFIX}')
print(f'Found {len(keys_by_dataset)} datasets under {PREFIX}')
for dataset_name in sorted(keys_by_dataset):
    print(
        f" - {dataset_name}: {len(keys_by_dataset[dataset_name])} parquet files, "
        f"{len(annotation_keys_by_dataset.get(dataset_name, []))} annotation JSON files"
    )


In [ ]:
if not parquet_keys:
    raise RuntimeError('No parquet files found under processed/250hz/. Run the pipeline first.')

dataset_s3_roots = {
    dataset_name: f"s3a://{S3_BUCKET}/{PREFIX}{dataset_name}/"
    for dataset_name in sorted(keys_by_dataset)
}

print('Dataset S3 roots (read directly; no local parquet cache download):')
for dataset_name, dataset_root in dataset_s3_roots.items():
    print(f' - {dataset_name}: {dataset_root}')


In [ ]:
from urllib.parse import urlparse

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

parsed_endpoint = urlparse(S3_ENDPOINT_URL)
s3a_endpoint = parsed_endpoint.netloc or parsed_endpoint.path
s3a_ssl_enabled = parsed_endpoint.scheme == 'https'

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('processed-250hz-eda')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint', s3a_endpoint)
    .config('spark.hadoop.fs.s3a.access.key', S3_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', S3_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', str(s3a_ssl_enabled).lower())
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .getOrCreate()
)

dataset_recording_dfs = {}
dataset_beat_annotation_dfs = {}
dataset_rhythm_episode_dfs = {}

for dataset_name, dataset_root in dataset_s3_roots.items():
    recording_df = (
        spark.read
        .option('recursiveFileLookup', 'true')
        .option('pathGlobFilter', '*.parquet')
        .parquet(dataset_root)
    )
    dataset_recording_dfs[dataset_name] = recording_df

    annotation_keys = annotation_keys_by_dataset.get(dataset_name, [])
    annotation_count = len(annotation_keys)
    if annotation_count > 0:
        annotation_paths = [f"s3a://{S3_BUCKET}/{key}" for key in annotation_keys]
        annotations_df = (
            spark.read
            .option('multiLine', 'true')
            .json(annotation_paths)
        )

        if set(annotations_df.columns) == {'_corrupt_record'}:
            print(
                f"Warning: annotation JSON could not be parsed for {dataset_name}; "
                "using empty beat/rhythm annotation DataFrames."
            )
            beat_df = spark.createDataFrame(
                [],
                'recording_id string, slice_index int, annotation_source string, target_sampling_rate int, '
                'original_sampling_rate int, beat_number int, beat_sample_index int, beat_label string',
            )
            rhythm_df = spark.createDataFrame(
                [],
                'recording_id string, slice_index int, annotation_source string, target_sampling_rate int, '
                'original_sampling_rate int, start_sample int, end_sample int, rhythm_type string, original_label string',
            )
        else:
            beat_df = (
                annotations_df
                .where(F.col('beat_annotations').isNotNull())
                .where(F.col('beat_annotations.positions').isNotNull())
                .select(
                    'recording_id',
                    'slice_index',
                    'annotation_source',
                    'target_sampling_rate',
                    'original_sampling_rate',
                    F.posexplode('beat_annotations.positions').alias('beat_number', 'beat_sample_index'),
                    F.col('beat_annotations.labels').alias('_beat_labels'),
                )
                .withColumn('beat_label', F.element_at(F.col('_beat_labels'), F.col('beat_number') + F.lit(1)))
                .drop('_beat_labels')
            )

            rhythm_df = (
                annotations_df
                .where(F.col('rhythm_episodes').isNotNull())
                .select(
                    'recording_id',
                    'slice_index',
                    'annotation_source',
                    'target_sampling_rate',
                    'original_sampling_rate',
                    F.explode('rhythm_episodes').alias('episode'),
                )
                .select(
                    'recording_id',
                    'slice_index',
                    'annotation_source',
                    'target_sampling_rate',
                    'original_sampling_rate',
                    F.col('episode.start_sample').alias('start_sample'),
                    F.col('episode.end_sample').alias('end_sample'),
                    F.col('episode.rhythm_type').alias('rhythm_type'),
                    F.col('episode.original_label').alias('original_label'),
                )
            )
    else:
        beat_df = spark.createDataFrame(
            [],
            'recording_id string, slice_index int, annotation_source string, target_sampling_rate int, '
            'original_sampling_rate int, beat_number int, beat_sample_index int, beat_label string',
        )
        rhythm_df = spark.createDataFrame(
            [],
            'recording_id string, slice_index int, annotation_source string, target_sampling_rate int, '
            'original_sampling_rate int, start_sample int, end_sample int, rhythm_type string, original_label string',
        )

    dataset_beat_annotation_dfs[dataset_name] = beat_df
    dataset_rhythm_episode_dfs[dataset_name] = rhythm_df

    # Convenience variable, e.g. df_mit_bih_atrial_fibrillation_database
    safe_name = dataset_name.replace('-', '_')
    globals()[f"df_{safe_name}"] = recording_df
    globals()[f"df_{safe_name}_recordings"] = recording_df
    globals()[f"df_{safe_name}_beat_annotations"] = beat_df
    globals()[f"df_{safe_name}_rhythm_episodes"] = rhythm_df

print(f'Built {len(dataset_recording_dfs)} dataset-specific DataFrames:')
for dataset_name, recording_df in dataset_recording_dfs.items():
    beat_df = dataset_beat_annotation_dfs[dataset_name]
    rhythm_df = dataset_rhythm_episode_dfs[dataset_name]
    print(
        f' - {dataset_name}: recordings={len(recording_df.columns)} cols, '
        f'beat_annotations={len(beat_df.columns)} cols, rhythm_episodes={len(rhythm_df.columns)} cols'
    )

for dataset_name, recording_df in dataset_recording_dfs.items():
    beat_df = dataset_beat_annotation_dfs[dataset_name]
    rhythm_df = dataset_rhythm_episode_dfs[dataset_name]

    print(f"\n=== Dataset: {dataset_name} ===")
    print('Recordings schema:')
    recording_df.printSchema()
    print('Recordings head (first 10 rows):')
    recording_df.show(10, truncate=False)
    print('Optional: rows per source parquet file (first 10):')
    (
        recording_df.withColumn('_source_file', F.input_file_name())
          .groupBy('_source_file')
          .count()
          .orderBy(F.desc('count'))
          .show(10, truncate=False)
    )

    print('Beat annotations schema:')
    beat_df.printSchema()
    print('Beat annotations head (first 10 rows):')
    beat_df.show(10, truncate=False)

    print('Rhythm episodes schema:')
    rhythm_df.printSchema()
    print('Rhythm episodes head (first 10 rows):')
    rhythm_df.show(10, truncate=False)
